# Training detector restoran (waiter / plate_food / customer) dengan YOLO11

**Jalankan di Google Colab**, bukan di Mac. Runtime → Change runtime type → **T4 GPU** (atau L4).

Cara pakai:
1. Simpan API key Roboflow di Colab Secrets (ikon kunci di sidebar kiri) dengan nama `ROBOFLOW_API_KEY`, aktifkan *Notebook access*. Atau ketik manual saat diminta.
2. Runtime → **Run all**.
3. Jika runtime terputus: Run all lagi dari atas. Dataset dibangun ulang dan **training lanjut otomatis** dari checkpoint di Google Drive.

Hasil: `MyDrive/cctv_resto/models/restaurant_detector.pt`

In [ ]:
# 1) Cek GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "GPU tidak aktif: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2) Install dependensi
!pip -q install ultralytics roboflow

In [ ]:
# 3) Mount Google Drive (checkpoint & hasil disimpan di sini agar aman jika runtime putus)
from google.colab import drive
drive.mount('/content/drive')

import shutil, random, yaml
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/cctv_resto')
MODELS, RUNS = ROOT / 'models', ROOT / 'runs'
for p in (MODELS, RUNS):
    p.mkdir(parents=True, exist_ok=True)
RUN_NAME = 'restaurant'
DATA = Path('/content/data')
RAW, MERGED = DATA / 'raw', DATA / 'merged'

In [ ]:
# 4) Roboflow API key (jangan ditulis di kode / dibagikan)
try:
    from google.colab import userdata
    API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    from getpass import getpass
    API_KEY = getpass('Roboflow API key: ')

In [ ]:
# 5) Dataset & skema class
# (workspace, project, all_to)
#   all_to=None          -> class dipetakan lewat KEYWORDS di bawah
#   all_to="plate_food"  -> SEMUA class dataset itu dijadikan plate_food
DATASETS = [
    ("akash-tt3fk", "ramada-bytez", None),
    ("insper-erh3j", "restaurant-tables-ymr9k", None),
    ("ktu", "plate-food", "plate_food"),
    ("restaurant-monitoring-system", "food_detection-kuscy", "plate_food"),
]
CLASSES = ["waiter", "plate_food", "customer"]
KEYWORDS = {  # cocokkan sebagian nama class (huruf kecil)
    "waiter": ["waiter", "waitress", "staff", "server"],
    "plate_food": ["plate", "food", "dish", "bowl", "tray", "meal"],
    "customer": ["customer", "person", "people", "guest", "diner"],
}
VERSION = None  # None = versi terbaru tiap dataset; isi angka untuk mengunci versi

In [ ]:
# 6) Download dataset (format yolov8 = txt label, kompatibel dengan YOLO11)
from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)

for ws, proj, _ in DATASETS:
    root = RAW / proj
    if (root / 'data.yaml').exists():
        print('sudah ada:', proj)
        continue
    project = rf.workspace(ws).project(proj)
    try:
        ver = VERSION or max(int(v.version) for v in project.versions())
    except Exception as e:
        print('gagal baca daftar versi, pakai versi 1:', e)
        ver = 1
    print(f'download {ws}/{proj} versi {ver} ...')
    project.version(ver).download('yolov8', location=str(root))

In [ ]:
# 7) Remap class -> skema kita, gabungkan, split train/valid/test
def map_name(name, all_to):
    if all_to:
        return all_to
    n = name.lower()
    for target in ("waiter", "plate_food", "customer"):  # waiter dulu agar 'staff person' tidak jadi customer
        if any(k in n for k in KEYWORDS[target]):
            return target
    return None

shutil.rmtree(MERGED, ignore_errors=True)
for split in ('train', 'valid', 'test'):
    for d in ('images', 'labels'):
        (MERGED / split / d).mkdir(parents=True, exist_ok=True)

stats = {c: 0 for c in CLASSES}
license_lines = []
for ws, proj, all_to in DATASETS:
    root = RAW / proj
    cfg = yaml.safe_load((root / 'data.yaml').read_text())
    names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][i] for i in sorted(cfg['names'])]
    mapping = {i: map_name(n, all_to) for i, n in enumerate(names)}
    print(f"\n[{proj}] class asli -> skema kita")
    for i, n in enumerate(names):
        print(f"   {n!r:32} -> {mapping[i]}")
    rb = cfg.get('roboflow', {})
    license_lines.append(f"- {ws}/{proj} (versi {rb.get('version')}): lisensi = {rb.get('license')} | {rb.get('url')}")

    for split in ('train', 'valid', 'test'):
        img_dir = root / split / 'images'
        if not img_dir.exists():
            continue
        for img in img_dir.iterdir():
            lbl = root / split / 'labels' / (img.stem + '.txt')
            if not lbl.exists():
                continue
            out = []
            for line in lbl.read_text().splitlines():
                p = line.split()
                if len(p) < 5:
                    continue
                target = mapping.get(int(float(p[0])))
                if target is None:
                    continue
                if len(p) > 5:  # polygon (segmentasi) -> bounding box
                    xs, ys = [float(v) for v in p[1::2]], [float(v) for v in p[2::2]]
                    x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
                    p = [p[0], (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]
                out.append(f"{CLASSES.index(target)} " + " ".join(f"{float(v):.6f}" for v in p[1:5]))
                stats[target] += 1
            if not out:
                continue
            name = f"{proj}_{img.name}"
            shutil.copy(img, MERGED / split / 'images' / name)
            (MERGED / split / 'labels' / (Path(name).stem + '.txt')).write_text("\n".join(out))

def n_imgs(split):
    return len(list((MERGED / split / 'images').iterdir()))

if n_imgs('valid') == 0:  # tidak ada split valid -> ambil 10% train
    random.seed(0)
    files = sorted((MERGED / 'train' / 'images').iterdir())
    for img in random.sample(files, max(1, len(files) // 10)):
        lbl = MERGED / 'train' / 'labels' / (img.stem + '.txt')
        shutil.move(str(img), MERGED / 'valid' / 'images' / img.name)
        shutil.move(str(lbl), MERGED / 'valid' / 'labels' / lbl.name)
TEST_SPLIT = 'test' if n_imgs('test') else 'valid'

(MERGED / 'data.yaml').write_text(yaml.safe_dump({
    'path': str(MERGED), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
    'names': {i: c for i, c in enumerate(CLASSES)},
}))
print("\njumlah objek per class:", stats)
print({s: n_imgs(s) for s in ('train', 'valid', 'test')}, "gambar | evaluasi akhir pakai split:", TEST_SPLIT)
for c, n in stats.items():
    if n == 0:
        print(f"PERINGATAN: class '{c}' tidak punya data. Cek pemetaan di atas (KEYWORDS / all_to).")
print("\nLISENSI DATASET (cek sebelum dipakai di luar riset):")
print("\n".join(license_lines))

In [ ]:
# 8) Training. Bisa dilanjutkan jika Colab terputus (checkpoint di Drive).
# Hyperparameter (T4 16 GB):
#   model  yolo11s.pt -> lebih akurat dari nano, masih cukup cepat untuk inference di Mac
#   epochs 80         -> cukup untuk fine-tune dari bobot COCO
#   patience 20       -> berhenti lebih awal jika mAP tidak naik 20 epoch
#   imgsz 640         -> sama dengan ukuran inference di Mac
#   batch 16          -> aman di T4; turunkan ke 8 jika muncul "CUDA out of memory"
#   workers 2         -> Colab hanya punya 2 CPU
from ultralytics import YOLO

last = RUNS / RUN_NAME / 'weights' / 'last.pt'

def finished(path):
    # Ultralytics membuang optimizer dari checkpoint setelah training selesai
    return torch.load(path, map_location='cpu', weights_only=False).get('optimizer') is None

if last.exists() and not finished(last):
    print('Checkpoint ditemukan, melanjutkan training...')
    YOLO(str(last)).train(resume=True)
elif last.exists():
    print('Training sebelumnya sudah selesai, lanjut ke evaluasi.')
else:
    YOLO('yolo11s.pt').train(
        data=str(MERGED / 'data.yaml'), epochs=80, imgsz=640, batch=16, patience=20,
        workers=2, device=0, project=str(RUNS), name=RUN_NAME, exist_ok=True, plots=True, seed=0,
    )

In [ ]:
# 9) Evaluasi: precision, recall, mAP (per class)
best = RUNS / RUN_NAME / 'weights' / 'best.pt'
model = YOLO(str(best))
m = model.val(data=str(MERGED / 'data.yaml'), split=TEST_SPLIT, imgsz=640, plots=True)
print(f"\nsplit evaluasi: {TEST_SPLIT}")
print(f"precision {m.box.mp:.3f} | recall {m.box.mr:.3f} | mAP50 {m.box.map50:.3f} | mAP50-95 {m.box.map:.3f}")
for i, c in enumerate(m.box.ap_class_index):
    p, r, ap50, ap = m.box.class_result(i)
    print(f"  {model.names[int(c)]:12} P={p:.3f}  R={r:.3f}  mAP50={ap50:.3f}  mAP50-95={ap:.3f}")

In [ ]:
# 10) Simpan ke Drive + export ONNX / CoreML
shutil.copy(best, MODELS / 'restaurant_detector.pt')
(MODELS / 'DATASET_LICENSES.md').write_text(
    "# Lisensi dataset detector restoran\n\n" + "\n".join(license_lines) + "\n")

onnx = model.export(format='onnx', imgsz=640)
shutil.copy(onnx, MODELS / 'restaurant_detector.onnx')
try:
    ml = model.export(format='coreml', imgsz=640)
    shutil.copytree(ml, MODELS / 'restaurant_detector.mlpackage', dirs_exist_ok=True)
except Exception as e:  # CoreML export di Linux kadang gagal; .pt sudah cukup untuk Mac
    print('Export CoreML dilewati:', e)

print("""
SELESAI. Cara pakai di Mac:
1. Buka drive.google.com > My Drive > cctv_resto > models
2. Download restaurant_detector.pt
3. Taruh di folder project:  cctv/models/restaurant_detector.pt
4. Jalankan seperti biasa (python -m rcis.cli --rtsp --display).
   Fitur waktu tunggu makanan & pembeda pelayan aktif otomatis jika file itu ada.
""")
print(sorted(p.name for p in MODELS.iterdir()))